In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the datasets
print("Loading datasets...")
review_train_x = pd.read_csv('review_train_x.csv')
review_test_x = pd.read_csv('review_test_x.csv')
review_train_y = pd.read_csv('review_train_y.csv')
business_data = pd.read_csv('business_data_cleaned.csv')

print(f"review_train_x shape: {review_train_x.shape}")
print(f"review_test_x shape: {review_test_x.shape}")
print(f"review_train_y shape: {review_train_y.shape}")
print(f"business_data shape: {business_data.shape}")
print("\nDatasets loaded successfully!")

Loading datasets...
review_train_x shape: (1577278, 6)
review_test_x shape: (700000, 6)
review_train_y shape: (1577278, 1)
business_data shape: (15000, 112)

Datasets loaded successfully!


In [2]:
# Step 1: Add label column to distinguish train and test data
print("Adding data source labels...")
review_train_x['data_source'] = 'train'
review_test_x['data_source'] = 'test'

print(f"review_train_x with label - shape: {review_train_x.shape}")
print(f"review_test_x with label - shape: {review_test_x.shape}")

# Step 2: Concatenate train and test data vertically
print("\nConcatenating train and test data...")
combined_reviews = pd.concat([review_train_x, review_test_x], axis=0, ignore_index=True)
del review_train_x, review_test_x  # Free memory
print(f"Combined reviews shape: {combined_reviews.shape}")
print(f"Data source distribution:\n{combined_reviews['data_source'].value_counts()}")

# Step 3: Select only numeric columns from business_data to reduce memory
print("\nPreparing business features...")
numeric_cols = ['business_id'] + business_data.select_dtypes(include=[np.number]).columns.tolist()
business_data_numeric = business_data[numeric_cols].copy()
print(f"Business data numeric columns selected: {len(numeric_cols)}")

# Merge combined data with business_data on business_id
print("\nMerging with business_data (numeric columns only)...")
merged_data = combined_reviews.merge(business_data_numeric, on='business_id', how='inner')
del combined_reviews, business_data_numeric  # Free memory
print(f"Merged data shape after business join: {merged_data.shape}")
print(f"Data source distribution after merge:\n{merged_data['data_source'].value_counts()}")

Adding data source labels...
review_train_x with label - shape: (1577278, 7)
review_test_x with label - shape: (700000, 7)

Concatenating train and test data...
Combined reviews shape: (2277278, 7)
Data source distribution:
data_source
train    1577278
test      700000
Name: count, dtype: int64

Preparing business features...
Business data numeric columns selected: 25

Merging with business_data (numeric columns only)...
Merged data shape after business join: (2277278, 31)
Data source distribution after merge:
data_source
train    1577278
test      700000
Name: count, dtype: int64


In [3]:
# Step 4: Resplit data back into train and test, and drop the label column
print("Splitting data back into train and test sets...")
train_data = merged_data[merged_data['data_source'] == 'train'].copy()
test_data = merged_data[merged_data['data_source'] == 'test'].copy()

# Drop the data_source column
train_data = train_data.drop('data_source', axis=1)
test_data = test_data.drop('data_source', axis=1)

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

# Step 5: Select features from business_data
print("\nPreparing features...")
business_features = business_data.select_dtypes(include=[np.number]).columns.tolist()
if 'is_open' in business_features:
    business_features.remove('is_open')

# Select features that exist in train/test data
feature_cols = [col for col in business_features if col in train_data.columns]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Extract features
X_train_full = train_data[feature_cols].fillna(0)
X_test_full = test_data[feature_cols].fillna(0)

print(f"\nX_train_full shape: {X_train_full.shape}")
print(f"X_test_full shape: {X_test_full.shape}")

Splitting data back into train and test sets...
Train data shape: (1577278, 30)
Test data shape: (700000, 30)

Preparing features...
Number of features: 22
Feature columns: ['latitude', 'longitude', 'review_count', 'RestaurantsPriceRange2', 'BusinessParking', 'GoodForMeal', 'Ambience', 'Music', 'HairSpecializesIn', 'BestNights', 'FoodorRestaurant', 'Services', 'SalonsandSpas', 'Entertainment', 'Travel', 'Shopping', 'HealthandFitness', 'Automotive', 'Pets', 'Activelife', 'Other', 'is_hours_missing']

X_train_full shape: (1577278, 22)
X_test_full shape: (700000, 22)


In [4]:
# Step 6: Concatenate review_train_y as target variable
print("Preparing target variable...")
y_train_full = review_train_y['top_useful'].values[:len(X_train_full)]

print(f"y_train_full shape: {y_train_full.shape}")
print(f"Target variable distribution for training data:")
print(f"Class 0: {(y_train_full == 0).sum()} ({(y_train_full == 0).sum() / len(y_train_full) * 100:.2f}%)")
print(f"Class 1: {(y_train_full == 1).sum()} ({(y_train_full == 1).sum() / len(y_train_full) * 100:.2f}%)")

print(f"\nX_train_full: {X_train_full.shape}")
print(f"X_test_full: {X_test_full.shape}")
print(f"y_train_full: {y_train_full.shape}")
print("\nData preparation complete!")

Preparing target variable...
y_train_full shape: (1577278,)
Target variable distribution for training data:
Class 0: 1386131 (87.88%)
Class 1: 191147 (12.12%)

X_train_full: (1577278, 22)
X_test_full: (700000, 22)
y_train_full: (1577278,)

Data preparation complete!


In [ ]:
# Step 7: Decision trees do not require feature scaling
print("Preparing decision tree inputs...")
X_train_tree = X_train_full
X_test_tree = X_test_full

print(f"X_train_tree shape: {X_train_tree.shape}")
print(f"X_test_tree shape: {X_test_tree.shape}")
print("\nDecision tree input preparation complete!")

Scaling features for SVM...
Scaled X_train shape: (1577278, 22)
Scaled X_test shape: (700000, 22)
Train features mean: [ 3.17719248e-16  1.18784329e-16 -5.76622955e-18 -1.55976509e-16
  0.00000000e+00]
Train features std: [1. 1. 1. 1. 0.]

Feature scaling complete!


In [ ]:
# Step 8: Train Decision Tree Model on full training data
print("Training Decision Tree Model on training data...")
print("This may take a few minutes...\n")
dt_model = DecisionTreeClassifier(random_state=42, max_depth=None, min_samples_split=10)
dt_model.fit(X_train_tree, y_train_full)

# Make predictions on test set
y_test_pred_proba = dt_model.predict_proba(X_test_tree)[:, 1]
y_test_pred = dt_model.predict(X_test_tree)

# Evaluate model on training set
y_train_pred_proba = dt_model.predict_proba(X_train_tree)[:, 1]
y_train_pred = dt_model.predict(X_train_tree)

train_score = dt_model.score(X_train_tree, y_train_full)

print("\n" + "=" * 60)
print("DECISION TREE MODEL PERFORMANCE")
print("=" * 60)
print(f"Train Accuracy: {train_score:.4f}")

# Additional metrics for test set predictions
print("\n" + "=" * 60)
print("TEST SET EVALUATION (Predictions)")
print("=" * 60)
print(f"Prediction distribution:")
print(f"Class 0 predictions: {(y_test_pred == 0).sum()} ({(y_test_pred == 0).sum() / len(y_test_pred) * 100:.2f}%)")
print(f"Class 1 predictions: {(y_test_pred == 1).sum()} ({(y_test_pred == 1).sum() / len(y_test_pred) * 100:.2f}%)")
print(f"\nProbability distribution (for Class 1):")
print(f"Mean probability: {y_test_pred_proba.mean():.4f}")
print(f"Std probability: {y_test_pred_proba.std():.4f}")
print("=" * 60)

Training SVM Model on training data...
This may take several minutes...



NameError: name 'SVC' is not defined

In [ ]:
# Step 9: Calculate ROC curve and TPR at 10% FPR on training data
print("Computing ROC Curve on Training Data...")

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_train_full, y_train_pred_proba)
roc_auc = roc_auc_score(y_train_full, y_train_pred_proba)

# Find TPR at 10% FPR
fpr_target = 0.10
idx = np.argmin(np.abs(fpr - fpr_target))
tpr_at_fpr = tpr[idx]
actual_fpr = fpr[idx]
threshold_at_fpr = thresholds[idx]

print("\n" + "=" * 60)
print("ROC CURVE ANALYSIS - TPR AT 10% FPR (Training Data)")
print("=" * 60)
print(f"Target FPR: {fpr_target:.4f}")
print(f"Actual FPR: {actual_fpr:.4f}")
print(f"TPR at ~10% FPR: {tpr_at_fpr:.4f}")
print(f"Threshold: {threshold_at_fpr:.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("=" * 60)

In [ ]:
# Step 10: Plot ROC Curve
print("Plotting ROC Curve...")
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(fpr, tpr, 'b-', label=f'ROC Curve (AUC={roc_auc:.4f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

# Highlight the point at 10% FPR
ax.plot(actual_fpr, tpr_at_fpr, 'ro', markersize=10, label=f'Point at ~10% FPR (TPR={tpr_at_fpr:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve - Decision Tree (Training Data)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print("ROC Curve plotted successfully!")

In [ ]:
# Step 11: Generate Learning Curve
print("Generating Learning Curve (this may take a while for Decision Tree)...")
print("Note: Using sample sizes for efficiency on large dataset\n")

# Use a smaller subset for learning curve if dataset is too large
sample_size = min(50000, len(X_train_tree))
sample_idx = np.random.choice(len(X_train_tree), sample_size, replace=False)
X_sample = X_train_tree.iloc[sample_idx]
y_sample = y_train_full[sample_idx]

train_sizes, train_scores, val_scores = learning_curve(
    DecisionTreeClassifier(random_state=42, max_depth=None, min_samples_split=10),
    X_sample, y_sample,
    cv=5,
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    verbose=1,
    scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

print("Learning curve completed!")

# Plot Learning Curve
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(train_sizes, train_mean, 'o-', color='r', label='Training Score', linewidth=2, markersize=6)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='r')

ax.plot(train_sizes, val_mean, 'o-', color='g', label='Validation Score', linewidth=2, markersize=6)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='g')

ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Learning Curve - Decision Tree', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Learning curve plotted successfully!")